# SWaT — Part 2: Detailed Model Evaluation & Visualisation
### Run this notebook AFTER `swat_ml_pipeline_v2.ipynb`

Covers every plot needed for a research paper or deployment report:
| # | Plot | File saved |
|---|------|-----------|
| 1 | Class distribution (counts + pie) | `01_class_dist.png` |
| 2 | Sensor time-series per attack | `02_sensor_timeseries.png` |
| 3 | Feature correlation heatmap | `03_corr_heatmap.png` |
| 4 | SMOTE before/after | `04_smote_balance.png` |
| 5 | Training curves (MLP + LSTM) | `05_training_curves.png` |
| 6 | Confusion matrices — all models | `06_cm_*.png` |
| 7 | ROC curves — binary models | `07_roc_binary.png` |
| 8 | One-vs-rest ROC — multi-class XGB | `08_roc_mc.png` |
| 9 | Precision-Recall curves | `09_pr_curves.png` |
| 10 | Per-class F1 comparison | `10_f1_comparison.png` |
| 11 | Feature importance (RF + XGB) | `11_feat_importance.png` |
| 12 | SHAP summary (XGB) | `12_shap_summary.png` |
| 13 | Temporal sensor windows (attack vs normal) | `13_temporal_windows.png` |
| 14 | Model comparison radar chart | `14_radar.png` |
| 15 | Save best model with full metadata | `best_model_bundle.joblib` |


## 0 · Setup — Load Everything from Part 1

In [ ]:
%pip install shap

In [ ]:
# !pip install shap scikit-learn xgboost joblib torch matplotlib seaborn pandas numpy

import warnings, json as _json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import shap

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn

warnings.filterwarnings('ignore')
np.random.seed(42)

MODELS_DIR = Path("saved_models")
PLOTS_DIR  = Path("plots"); PLOTS_DIR.mkdir(exist_ok=True)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ATTACK_NAMES = {
    0:"Normal Operation", 1:"Tank Overflow Attack", 2:"Chemical Depletion Attack",
    3:"Membrane Damage Attack", 4:"pH Manipulation Attack", 5:"Slow Ramp Attack",
    6:"Reconnaissance Scan", 7:"Denial of Service", 8:"Replay Attack",
    9:"Valve Manipulation Attack",
}
ID_REMAP = {0:0, 8:1, 9:2, 10:3, 11:4, 12:5, 13:6, 14:7, 15:8, 16:9}
NUM_CLASSES = 10
SHORT = [f"[{i}]\n{ATTACK_NAMES[i][:12]}" for i in range(10)]
SHORT_FLAT = [f"[{i}] {ATTACK_NAMES[i][:16]}" for i in range(10)]

# ── Colour palette ────────────────────────────────────────────────────────────
PALETTE = ['#1565C0','#E53935','#2E7D32','#F57F17','#6A1B9A',
           '#00838F','#AD1457','#37474F','#558B2F','#4E342E']
MC_CMAP = dict(zip(range(10), PALETTE))

print("Setup complete — loading saved models...")


In [ ]:
# ── Load models ───────────────────────────────────────────────────────────────
scaler    = joblib.load(MODELS_DIR / "scaler.joblib")
rf_bin    = joblib.load(MODELS_DIR / "rf_binary.joblib")
xgb_bin   = joblib.load(MODELS_DIR / "xgb_binary.joblib")
svm_bin   = joblib.load(MODELS_DIR / "svm_binary.joblib")
rf_mc     = joblib.load(MODELS_DIR / "rf_multiclass.joblib")
xgb_mc    = joblib.load(MODELS_DIR / "xgb_multiclass.joblib")

with open(MODELS_DIR / "pipeline_metadata.json") as f:
    meta = _json.load(f)
FEATURE_COLS = meta['feature_cols']
print(f"Loaded models. Features: {len(FEATURE_COLS)}")

# ── Load and prepare dataset  ─────────────────────────────────────────────────
DATA_PATH = "compressed_data_csv.gz"
df_raw = pd.read_csv(DATA_PATH, compression='gzip')
df_raw['ATTACK_ID']   = df_raw['ATTACK_ID'].map(ID_REMAP)
df_raw['ATTACK_NAME'] = df_raw['ATTACK_ID'].map(ATTACK_NAMES)
df_raw['ts']          = pd.to_datetime(df_raw['Timestamp'])
df_raw = df_raw.sort_values(['run_id','elapsed_seconds']).reset_index(drop=True)
print(f"Dataset: {df_raw.shape}")


In [ ]:
# ── Rebuild clean feature matrix (same pipeline as Part 1) ───────────────────
DEAD_COLS = [
    'DPIT_301','FIT_301','UF_Last_Backwash','Turbidity_UF',
    'AIT_501','AIT_502','AIT_503','AIT_504',
    'FIT_501','FIT_502','FIT_503','FIT_504',
    'PIT_501','PIT_502','PIT_503',
    'RO_Runtime','RO_Fouling_Factor','RO_Last_Cleaning',
    'TDS_Feed','TDS_Permeate','FIT_601',
    'Energy_P101','Energy_P301','Energy_P501','Energy_Total',
    'UF_Runtime','UF_Last_Backwash',
    'P_203','MV_303','MV_304','P_301','UF_Backwash_Active',
    'P_403','P_501','RO_Cleaning_Active','P_601','P_602','P_603',
    'High_Fouling_Alarm','System_Run','UV_401',
    'Timestamp','ATTACK_NAME','MITRE_ID','ts',
]
DEAD_COLS = [c for c in DEAD_COLS if c in df_raw.columns]
df = df_raw.drop(columns=DEAD_COLS).copy()

bool_cols = [c for c in df.columns if df[c].dtype == bool]
df[bool_cols] = df[bool_cols].astype(np.int8)

# Tank depletion normalisation
for col in ['Acid_Tank_Level','Chlorine_Tank_Level',
            'Coagulant_Tank_Level','Bisulfate_Tank_Level']:
    if col not in df.columns: continue
    rs = df[df.ATTACK_ID==0].groupby('run_id')[col].first().rename('_s')
    df = df.join(rs, on='run_id')
    df[f'{col}_pct'] = ((df['_s']-df[col])/df['_s'].clip(lower=1)).clip(0,1)
    df.drop(columns=['_s'], inplace=True)

# Temporal features
WINDOW_SIZES     = [10,25,50]
TEMPORAL_SENSORS = [c for c in ['LIT_101','LIT_301','LIT_401','AIT_202',
                                  'FIT_101','FIT_401','Acid_Tank_Level',
                                  'Chlorine_Residual'] if c in df.columns]
parts = []
for rid, grp in df.groupby('run_id'):
    grp = grp.sort_values('elapsed_seconds').copy().reset_index(drop=True)
    dt  = grp['elapsed_seconds'].diff().fillna(0.2).clip(lower=0.01)
    for s in TEMPORAL_SENSORS:
        col_v = grp[s].astype(float)
        grp[f'd_{s}_dt'] = col_v.diff().fillna(0) / dt
        for w in WINDOW_SIZES:
            grp[f'{s}_rmean{w}'] = col_v.rolling(w, min_periods=1).mean()
            grp[f'{s}_rstd{w}']  = col_v.rolling(w, min_periods=1).std().fillna(0)
        base = col_v.expanding(min_periods=1).mean()
        cs   = (col_v - base).cumsum()
        grp[f'{s}_cusum'] = cs - cs.iloc[0]
    new_c = [f for s in TEMPORAL_SENSORS for f in df.columns
             if f.startswith((f'd_{s}',f'{s}_rmean',f'{s}_rstd',f'{s}_cusum'))
             if f in grp.columns]
    for c in new_c:
        if grp[c].isna().any():
            grp[c] = grp[c].ffill().bfill().fillna(0)
    parts.append(grp)
df = pd.concat(parts, ignore_index=True)

NON_FEAT = {'ATTACK_ID','run_id','elapsed_seconds'}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEAT
                and df[c].dtype in (np.float64,np.float32,np.int64,
                                    np.int32,np.int16,np.int8,float,int)]

X_raw = df[FEATURE_COLS].copy()
X_raw['_run'] = df['run_id'].values
X_filled = (X_raw.groupby('_run', group_keys=False)
            .apply(lambda g: g.ffill().bfill())
            .fillna(0)).drop(columns=['_run'])
X = X_filled.values.astype(np.float32)
y = df['ATTACK_ID'].values.astype(np.int64)
y_bin = (y > 0).astype(np.int64)

X_tr, X_te, y_tr, y_te, yb_tr, yb_te = train_test_split(
    X, y, y_bin, test_size=0.2, random_state=42, stratify=y)
X_tr_sc = scaler.transform(X_tr)
X_te_sc  = scaler.transform(X_te)

assert np.isnan(X_te_sc).sum() == 0
print(f"Feature matrix: {X.shape}  |  NaN: {np.isnan(X).sum()}  |  Test: {X_te_sc.shape}")


## 1 · Class Distribution

In [ ]:
counts = df.groupby('ATTACK_ID').size()

fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Bar chart
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.bar(range(NUM_CLASSES), counts.values,
               color=[PALETTE[i] for i in range(NUM_CLASSES)],
               edgecolor='white', linewidth=0.6)
ax1.set_xticks(range(NUM_CLASSES))
ax1.set_xticklabels([ATTACK_NAMES[i].replace(' ','
') for i in range(NUM_CLASSES)],
                     fontsize=7.5)
ax1.set_title("Row Count per Attack Class", fontweight='bold', fontsize=12)
ax1.set_ylabel("Rows"); ax1.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, v+200, f"{v:,}",
             ha='center', va='bottom', fontsize=7.5, fontweight='bold')

# Pie
ax2 = fig.add_subplot(gs[0, 2])
wedges, texts, autotexts = ax2.pie(
    counts.values,
    labels=[f"[{i}]" for i in range(NUM_CLASSES)],
    colors=PALETTE, autopct='%1.1f%%', startangle=140,
    textprops={'fontsize': 7.5})
for at in autotexts: at.set_fontsize(6.5)
legend_patches = [mpatches.Patch(color=PALETTE[i],
                  label=f"[{i}] {ATTACK_NAMES[i]}") for i in range(NUM_CLASSES)]
ax2.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1,1),
           fontsize=6.5, framealpha=0.7)
ax2.set_title("Distribution", fontweight='bold')

plt.suptitle("SWaT Dataset — Attack Class Distribution (117,695 rows, 4 runs)",
             fontweight='bold', fontsize=13, y=1.01)
plt.savefig(PLOTS_DIR/"01_class_dist.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Normal:Attack ratio = {counts[0]/counts[1:].sum():.2f}:1")


## 2 · Sensor Time-Series per Attack

In [ ]:
KEY_SENSORS = ['LIT_101','LIT_301','LIT_401','AIT_202',
               'Acid_Tank_Level','Chlorine_Residual','FIT_401','MV_101']
ATTACK_PLOT = {1:"Tank Overflow", 2:"Chemical Depletion",
               4:"pH Manipulation", 5:"Slow Ramp",
               9:"Valve Manipulation"}

fig, axes = plt.subplots(len(KEY_SENSORS), 1, figsize=(16, 22), sharex=False)
fig.subplots_adjust(hspace=0.55)

# Use run_01 data for a continuous time-series view
r1 = df[df['run_id']==1].sort_values('elapsed_seconds').reset_index(drop=True)
time_axis = r1['elapsed_seconds'].values / 60   # minutes

for ax, sensor in zip(axes, KEY_SENSORS):
    if sensor not in r1.columns: continue
    vals = r1[sensor].values.astype(float)

    # Background colour bands per attack
    prev_end = 0
    for _, seg in r1.groupby((r1['ATTACK_ID'] != r1['ATTACK_ID'].shift()).cumsum()):
        aid   = seg['ATTACK_ID'].iloc[0]
        t_st  = seg['elapsed_seconds'].iloc[0] / 60
        t_en  = seg['elapsed_seconds'].iloc[-1] / 60
        alpha = 0.0 if aid == 0 else 0.18
        if aid != 0:
            ax.axvspan(t_st, t_en, alpha=alpha, color=PALETTE[aid], label=f"[{aid}]")

    ax.plot(time_axis, vals, color='#212121', lw=0.7, alpha=0.85)
    ax.set_ylabel(sensor, fontsize=8, fontweight='bold')
    ax.set_xlabel("Time (minutes)", fontsize=7)
    ax.grid(alpha=0.25)

    # Annotate attack windows
    for aid, name in ATTACK_PLOT.items():
        seg = r1[r1['ATTACK_ID']==aid]
        if len(seg)==0: continue
        tm = seg['elapsed_seconds'].mean()/60
        ax.axvline(seg['elapsed_seconds'].iloc[0]/60,  color=PALETTE[aid], lw=1.2, ls='--', alpha=0.7)
        ax.axvline(seg['elapsed_seconds'].iloc[-1]/60, color=PALETTE[aid], lw=1.2, ls='--', alpha=0.7)
        ax.text(tm, ax.get_ylim()[1]*0.92, name[:8], fontsize=6,
                color=PALETTE[aid], ha='center', fontweight='bold')

plt.suptitle("Sensor Time-Series — Run 01  (shaded = attack windows)",
             fontweight='bold', fontsize=13, y=1.002)
plt.savefig(PLOTS_DIR/"02_sensor_timeseries.png", dpi=150, bbox_inches='tight')
plt.show()


## 3 · Correlation Heatmap (Top Features)

In [ ]:
# Correlation between top-30 features
top30_feats = (pd.DataFrame(X_tr_sc, columns=FEATURE_COLS)
               .corrwith(pd.Series(y_tr.astype(float)))
               .abs().sort_values(ascending=False)
               .head(30).index.tolist())

corr_mat = pd.DataFrame(X_te_sc, columns=FEATURE_COLS)[top30_feats].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, mask=mask, cmap='coolwarm', center=0,
            annot=False, linewidths=0.3, ax=ax,
            cbar_kws={"shrink":0.8, "label":"Pearson r"})
ax.set_title("Feature Correlation Matrix — Top 30 Features by Attack Label Correlation",
             fontweight='bold', fontsize=12)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"03_corr_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()


## 4 · SMOTE Before / After

In [ ]:
from imblearn.over_sampling import SMOTE

before_counts = Counter(y_tr)
smote = SMOTE(random_state=42, k_neighbors=3,
              sampling_strategy={2:4000, 3:4000, 7:4000})
X_bal, y_bal = smote.fit_resample(X_tr_sc, y_tr)
y_bin_bal    = (y_bal > 0).astype(np.int64)
after_counts  = Counter(y_bal)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = list(range(NUM_CLASSES))
bef = [before_counts.get(c,0) for c in classes]
aft = [after_counts.get(c,0)  for c in classes]
x   = np.arange(NUM_CLASSES)
w   = 0.38

axes[0].bar(x-w/2, bef, w, label='Before SMOTE', color='#90CAF9', edgecolor='#1565C0', lw=0.8)
axes[0].bar(x+w/2, aft, w, label='After SMOTE',  color='#EF9A9A', edgecolor='#C62828', lw=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"[{i}]" for i in classes], fontsize=9)
axes[0].set_title("SMOTE — Training Set Class Counts", fontweight='bold')
axes[0].set_ylabel("Samples"); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
for xi, (b, a) in enumerate(zip(bef, aft)):
    if a != b:
        axes[0].annotate(f"+{a-b:,}", xy=(xi+w/2, a), ha='center', va='bottom',
                         fontsize=7, color='#C62828')

# Pie — after
axes[1].pie(aft, labels=[f"[{i}]" for i in classes], colors=PALETTE,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize':7.5})
axes[1].set_title("After SMOTE — Distribution", fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR/"04_smote_balance.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Training set after SMOTE: {len(X_bal):,} samples")


## 5 · Training Curves (MLP + LSTM)

In [ ]:
# ── Re-train MLP quickly and capture history ──────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset

class SWaTMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def make_loader(Xtr, ytr, Xte, yte, batch=1024):
    tr = DataLoader(TensorDataset(torch.FloatTensor(Xtr),torch.LongTensor(ytr)),
                    batch_size=batch, shuffle=True,  num_workers=0)
    te = DataLoader(TensorDataset(torch.FloatTensor(Xte),torch.LongTensor(yte)),
                    batch_size=batch, shuffle=False, num_workers=0)
    return tr, te

def run_training(model, tr_loader, te_loader, epochs, crit, opt, sched=None, device=DEVICE):
    hist = {'tr_loss':[],'va_loss':[],'tr_acc':[],'va_acc':[],'va_f1':[]}
    best_f1, best_preds = 0., None
    for ep in range(1, epochs+1):
        model.train()
        tl, tc, tn = 0., 0, 0
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            out  = model(Xb); loss = crit(out, yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step()
            tl += loss.item()*len(yb); tc += (out.argmax(1)==yb).sum().item(); tn += len(yb)
        model.eval(); vl, vc, vn = 0., 0, 0; all_p, all_t = [], []
        with torch.no_grad():
            for Xb, yb in te_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                out = model(Xb); vl += crit(out,yb).item()*len(yb)
                p = out.argmax(1); vc += (p==yb).sum().item(); vn += len(yb)
                all_p.extend(p.cpu().numpy()); all_t.extend(yb.cpu().numpy())
        vf = f1_score(all_t, all_p, average='macro', zero_division=0)
        for k,v in zip(['tr_loss','va_loss','tr_acc','va_acc','va_f1'],
                       [tl/tn, vl/vn, tc/tn, vc/vn, vf]):
            hist[k].append(v)
        if vf > best_f1:
            best_f1 = vf; best_preds = np.array(all_p)
            torch.save(model.state_dict(), MODELS_DIR/f"mlp_multiclass_best.pt")
        if sched: sched.step(vl/vn)
    return hist, best_f1, best_preds

cw_t = compute_class_weight('balanced', classes=np.unique(y_bal), y=y_bal)
cw_mc_dict = dict(zip(np.unique(y_bal).tolist(), cw_t.tolist()))
cw_tensor  = torch.FloatTensor([cw_mc_dict.get(i,1.) for i in range(NUM_CLASSES)]).to(DEVICE)

mlp_mc = SWaTMLP(X_bal.shape[1], [512,256,128,64], NUM_CLASSES).to(DEVICE)
tr_l, te_l = make_loader(X_bal, y_bal, X_te_sc, y_te)
opt_mc = torch.optim.Adam(mlp_mc.parameters(), lr=1e-3, weight_decay=1e-5)
sch_mc = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_mc, patience=5, factor=0.5, verbose=False)
crit_mc = nn.CrossEntropyLoss(weight=cw_tensor)

print("Training MLP (50 epochs)...")
hist_mc, best_f1_mc, best_preds_mc = run_training(
    mlp_mc, tr_l, te_l, 50, crit_mc, opt_mc, sch_mc)
print(f"Best MLP F1: {best_f1_mc:.4f}")


In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
ep_x = range(1, len(hist_mc['tr_loss'])+1)
specs = [
    ('tr_loss','va_loss',   'Loss',        'blue','red'),
    ('tr_acc', 'va_acc',    'Accuracy',    'blue','red'),
    ('va_f1',  None,        'Macro F1',    'green', None),
]
for col, (tk, vk, title, tc, vc) in enumerate(specs):
    ax = axes[0, col]
    ax.plot(ep_x, hist_mc[tk], color=tc, lw=1.8, label='Train')
    if vk:
        ax.plot(ep_x, hist_mc[vk], color=vc, lw=1.8, label='Val', linestyle='--')
    if tk == 'va_f1':
        ax.axhline(best_f1_mc, color='grey', ls=':', label=f'Best={best_f1_mc:.4f}')
    ax.set_title(f"MLP — {title}", fontweight='bold')
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Learning rate schedule visualisation
ax_lr = axes[1, 0]
lrs = [opt_mc.param_groups[0]['lr']] * len(ep_x)   # approximate (captured at end)
ax_lr.plot(ep_x, hist_mc['va_loss'], color='purple', lw=1.8)
ax_lr.set_title("MLP Val Loss (ReduceLROnPlateau)", fontweight='bold')
ax_lr.set_xlabel("Epoch"); ax_lr.grid(alpha=0.3)

# Smoothed F1
from scipy.ndimage import uniform_filter1d
ax_sm = axes[1, 1]
smooth = uniform_filter1d(hist_mc['va_f1'], size=5)
ax_sm.plot(ep_x, hist_mc['va_f1'], alpha=0.35, color='green', lw=1)
ax_sm.plot(ep_x, smooth, color='green', lw=2.2, label='Smoothed F1')
ax_sm.axhline(best_f1_mc, color='grey', ls=':', label=f'Best={best_f1_mc:.4f}')
ax_sm.set_title("Smoothed Val Macro F1", fontweight='bold')
ax_sm.legend(fontsize=8); ax_sm.set_xlabel("Epoch"); ax_sm.grid(alpha=0.3)

# Loss gap (overfit indicator)
ax_gap = axes[1, 2]
gap = np.array(hist_mc['va_loss']) - np.array(hist_mc['tr_loss'])
ax_gap.fill_between(ep_x, 0, gap, where=gap>0, alpha=0.4, color='red',  label='Overfit')
ax_gap.fill_between(ep_x, 0, gap, where=gap<0, alpha=0.4, color='blue', label='Underfit')
ax_gap.plot(ep_x, gap, color='black', lw=1)
ax_gap.axhline(0, color='grey', lw=0.8)
ax_gap.set_title("Val Loss − Train Loss (Generalisation Gap)", fontweight='bold')
ax_gap.legend(fontsize=8); ax_gap.set_xlabel("Epoch"); ax_gap.grid(alpha=0.3)

plt.suptitle("MLP Training Diagnostics", fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"05_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


## 6 · Confusion Matrices — All Models

In [ ]:
def plot_cm(y_true, y_pred, title, fname, cmap='Blues'):
    cm   = confusion_matrix(y_true, y_pred, normalize='true')
    fig, axes = plt.subplots(1, 2, figsize=(20, 8),
                              gridspec_kw={'width_ratios':[3,1]})
    sns.heatmap(cm, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=SHORT_FLAT, yticklabels=SHORT_FLAT,
                ax=axes[0], linewidths=0.4, annot_kws={'size':8})
    axes[0].set_title(title, fontweight='bold', fontsize=12)
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].tick_params(axis='x', rotation=45, labelsize=8)
    axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

    # Per-class accuracy bar
    per_class_acc = cm.diagonal()
    axes[1].barh(SHORT_FLAT[::-1], per_class_acc[::-1],
                 color=[PALETTE[i] for i in range(NUM_CLASSES)][::-1])
    axes[1].set_xlim(0, 1.1)
    axes[1].set_title("Per-class\nRecall", fontweight='bold', fontsize=10)
    axes[1].set_xlabel("Recall")
    for i, v in enumerate(per_class_acc[::-1]):
        axes[1].text(v+0.01, i, f"{v:.2f}", va='center', fontsize=8)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR/fname, dpi=150, bbox_inches='tight')
    plt.show()
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    print(f"  Accuracy={acc:.4f}  Macro F1={f1:.4f}")

# Re-generate predictions
rf_preds  = rf_mc.predict(X_te_sc)
xgb_preds = xgb_mc.predict(X_te_sc)

plot_cm(y_te, best_preds_mc, "MLP Multi-class Confusion Matrix",    "06_cm_mlp.png",   cmap='Greens')
plot_cm(y_te, rf_preds,       "Random Forest Multi-class CM",        "06_cm_rf.png",    cmap='Blues')
plot_cm(y_te, xgb_preds,      "XGBoost Multi-class CM",              "06_cm_xgb.png",   cmap='Oranges')


In [ ]:
# ── Binary confusion matrices ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (mdl, name, cmap_) in zip(axes, [
    (rf_bin,  "RF Binary",  'Blues'),
    (xgb_bin, "XGB Binary", 'Oranges'),
    (svm_bin, "SVM Binary", 'Purples'),
]):
    preds = mdl.predict(X_te_sc)
    cm    = confusion_matrix(yb_te, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.3f', cmap=cmap_,
                xticklabels=['Normal','Attack'],
                yticklabels=['Normal','Attack'], ax=ax,
                linewidths=1, annot_kws={'size':13})
    f1 = f1_score(yb_te, preds, average='macro', zero_division=0)
    ax.set_title(f"{name}\nMacro F1={f1:.4f}", fontweight='bold')
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle("Binary Detection — Confusion Matrices", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"06_cm_binary.png", dpi=150, bbox_inches='tight')
plt.show()


## 7 · ROC Curves — Binary Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
aucs = {}
for ax, (mdl, name, col) in zip(axes, [
    (rf_bin,  "Random Forest", '#1565C0'),
    (xgb_bin, "XGBoost",       '#2E7D32'),
    (svm_bin, "SVM RBF",       '#E65100'),
]):
    proba = mdl.predict_proba(X_te_sc)[:,1]
    fpr, tpr, thresholds = roc_curve(yb_te, proba)
    auc   = roc_auc_score(yb_te, proba)
    aucs[name] = auc
    ax.plot(fpr, tpr, lw=2.5, color=col, label=f"AUC = {auc:.4f}")
    ax.plot([0,1],[0,1],'--', color='grey', lw=1)
    # Optimal threshold (Youden's J)
    j_idx = np.argmax(tpr - fpr)
    ax.scatter(fpr[j_idx], tpr[j_idx], marker='o', color='red', s=80, zorder=5,
               label=f"Best thr={thresholds[j_idx]:.3f}")
    ax.fill_between(fpr, tpr, alpha=0.08, color=col)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)

plt.suptitle("ROC Curves — Binary Anomaly Detection", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"07_roc_binary.png", dpi=150, bbox_inches='tight')
plt.show()
print("AUCs:", {k: f"{v:.4f}" for k,v in aucs.items()})


## 8 · One-vs-Rest ROC — Multi-class XGBoost

In [ ]:
y_ohe   = label_binarize(y_te, classes=list(range(NUM_CLASSES)))
xgb_prb = xgb_mc.predict_proba(X_te_sc)
rf_prb  = rf_mc.predict_proba(X_te_sc)

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for i, ax in enumerate(axes.flat):
    for prb, mname, col, ls in [
        (xgb_prb, "XGB",   '#E65100', '-'),
        (rf_prb,  "RF",    '#1565C0', '--'),
    ]:
        fpr, tpr, _ = roc_curve(y_ohe[:,i], prb[:,i])
        auc = roc_auc_score(y_ohe[:,i], prb[:,i])
        ax.plot(fpr, tpr, lw=2, color=col, ls=ls, label=f"{mname} {auc:.3f}")
    ax.plot([0,1],[0,1],':', color='grey', lw=1)
    ax.fill_between(fpr, tpr, alpha=0.07, color=PALETTE[i])
    ax.set_title(f"[{i}] {ATTACK_NAMES[i][:18]}", fontsize=8, fontweight='bold',
                 color=PALETTE[i])
    ax.legend(fontsize=7, loc='lower right')
    ax.set_xlabel("FPR", fontsize=7); ax.set_ylabel("TPR", fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle("One-vs-Rest ROC Curves — XGBoost vs Random Forest",
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"08_roc_mc.png", dpi=150, bbox_inches='tight')
plt.show()


## 9 · Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for i, ax in enumerate(axes.flat):
    for prb, mname, col in [
        (xgb_prb, "XGB", '#E65100'),
        (rf_prb,  "RF",  '#1565C0'),
    ]:
        prec, rec, _ = precision_recall_curve(y_ohe[:,i], prb[:,i])
        ap = average_precision_score(y_ohe[:,i], prb[:,i])
        ax.plot(rec, prec, lw=2, color=col, label=f"{mname} AP={ap:.3f}")
    # Baseline (random)
    base = y_ohe[:,i].mean()
    ax.axhline(base, color='grey', ls=':', lw=1, label=f"Baseline={base:.3f}")
    ax.fill_between(rec, prec, base, alpha=0.07, color=PALETTE[i])
    ax.set_title(f"[{i}] {ATTACK_NAMES[i][:18]}", fontsize=8, fontweight='bold',
                 color=PALETTE[i])
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xlabel("Recall", fontsize=7); ax.set_ylabel("Precision", fontsize=7)
    ax.set_xlim(0,1); ax.set_ylim(0,1.05); ax.grid(alpha=0.3)

plt.suptitle("Precision-Recall Curves — Multi-class",
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"09_pr_curves.png", dpi=150, bbox_inches='tight')
plt.show()


## 10 · Per-Class F1 Comparison

In [ ]:
model_preds = {
    'Random Forest': rf_preds,
    'XGBoost':       xgb_preds,
    'MLP':           best_preds_mc,
}
f1_rows = {}
for mname, preds in model_preds.items():
    rpt = classification_report(y_te, preds, output_dict=True, zero_division=0)
    f1_rows[mname] = {ATTACK_NAMES[i]: rpt.get(str(i),{}).get('f1-score',0.)
                      for i in range(NUM_CLASSES)}
f1_df = pd.DataFrame(f1_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar
x   = np.arange(NUM_CLASSES)
w   = 0.26
col3 = ['#1565C0','#E65100','#2E7D32']
for k, (mname, col) in enumerate(zip(f1_rows.keys(), col3)):
    axes[0].bar(x + k*w - w, f1_df[mname].values, w,
                label=mname, color=col, alpha=0.85, edgecolor='white', lw=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"[{i}]" for i in range(NUM_CLASSES)], fontsize=9)
axes[0].set_ylim(0, 1.12); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_title("Per-Class F1 Score — All Models", fontweight='bold')
axes[0].set_ylabel("F1 Score"); axes[0].legend()

# Heatmap
sns.heatmap(f1_df.T, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, ax=axes[1],
            xticklabels=[ATTACK_NAMES[i][:14] for i in range(NUM_CLASSES)],
            linewidths=0.5, annot_kws={'size':8})
axes[1].set_title("F1 Heatmap", fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=7.5)
axes[1].tick_params(axis='y', rotation=0,  labelsize=9)

plt.suptitle("Per-Class F1 Comparison", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"10_f1_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print(f1_df.round(4).to_string())


## 11 · Feature Importance — RF + XGBoost

In [ ]:
rf_imp  = pd.Series(rf_mc.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_mc.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
TOP_N = 30

fig, axes = plt.subplots(1, 2, figsize=(18, 11))
for ax, imp, name, cmap_name in [
    (axes[0], rf_imp,  "Random Forest", 'Blues_r'),
    (axes[1], xgb_imp, "XGBoost",       'Oranges_r'),
]:
    top = imp.head(TOP_N)
    colors = plt.cm.get_cmap(cmap_name)(np.linspace(0.3, 0.9, TOP_N))[::-1]
    ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
    ax.set_title(f"Top {TOP_N} Features — {name}", fontweight='bold', fontsize=11)
    ax.set_xlabel("Importance")
    for i, (feat, val) in enumerate(zip(top.index[::-1], top.values[::-1])):
        ax.text(val+0.0005, i, f"{val:.4f}", va='center', fontsize=7)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle("Feature Importance Comparison", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"11_feat_importance.png", dpi=150, bbox_inches='tight')
plt.show()

# Overlap: features in both top-20
top20_rf  = set(rf_imp.head(20).index)
top20_xgb = set(xgb_imp.head(20).index)
print(f"Top-20 overlap (RF ∩ XGB): {len(top20_rf & top20_xgb)} features")
print(sorted(top20_rf & top20_xgb))


## 12 · SHAP Summary Plot (XGBoost)

In [ ]:
# SHAP on a stratified subsample (1500 rows) for speed
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=1500, random_state=42)
_, shap_idx = next(sss.split(X_te_sc, y_te))
X_shap = X_te_sc[shap_idx]

explainer   = shap.TreeExplainer(xgb_mc)
shap_values = explainer.shap_values(X_shap)   # shape: (n_classes, n_samples, n_features)

# ── Plot 1: Mean |SHAP| across all classes ────────────────────────────────────
mean_abs_shap = np.abs(np.array(shap_values)).mean(axis=(0,1))
shap_imp = pd.Series(mean_abs_shap, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 8))
top25 = shap_imp.head(25)
colors = plt.cm.RdYlGn(top25.values / top25.values.max())
ax.barh(top25.index[::-1], top25.values[::-1], color=colors)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("SHAP Feature Importance — XGBoost Multi-class\n"
             "(averaged across all 10 classes)", fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR/"12_shap_bar.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: SHAP beeswarm for each attack class ───────────────────────────────
top10_feats = shap_imp.head(10).index.tolist()
feat_idx    = [FEATURE_COLS.index(f) for f in top10_feats]

fig, axes = plt.subplots(2, 5, figsize=(22, 10))
for i, ax in enumerate(axes.flat):
    sv = shap_values[i][:, feat_idx]   # (n_samples, 10)
    XV = X_shap[:, feat_idx]
    for j, feat in enumerate(top10_feats):
        sv_j = sv[:, j]
        xv_j = XV[:, j]
        # Scatter coloured by feature value
        sc = ax.scatter(sv_j, [j]*len(sv_j), c=xv_j, cmap='coolwarm',
                        alpha=0.3, s=4, vmin=np.percentile(xv_j,5),
                        vmax=np.percentile(xv_j,95))
    ax.set_yticks(range(len(top10_feats)))
    ax.set_yticklabels([f[:16] for f in top10_feats], fontsize=7)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f"[{i}] {ATTACK_NAMES[i][:16]}", fontsize=8,
                 fontweight='bold', color=PALETTE[i])
    ax.set_xlabel("SHAP", fontsize=7); ax.grid(alpha=0.2)

plt.suptitle("SHAP Beeswarm — Per Attack Class (top 10 features, colour = feature value)",
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"12_shap_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()


## 13 · Temporal Window Analysis

In [ ]:
# Compare rolling statistics for key sensors: Normal vs each attack
WINDOW = 25   # 5 seconds
fig = plt.figure(figsize=(18, 20))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.5, wspace=0.35)

plot_specs = [
    ('LIT_401',          'LIT_401_rmean25',  'LIT_401_rstd25',  [1, 5], "LIT_401"),
    ('AIT_202',          'AIT_202_rmean25',  'AIT_202_rstd25',  [4],    "AIT_202 (pH)"),
    ('Acid_Tank_Level',  'Acid_Tank_Level_rmean25','Acid_Tank_Level_rstd25',[2],"Acid Tank"),
    ('FIT_401',          'FIT_401_rmean25',  'FIT_401_rstd25',  [9],    "FIT_401"),
]

for plot_i, (raw_col, mean_col, std_col, attack_ids, label) in enumerate(plot_specs):
    row, col_ = divmod(plot_i, 2)
    ax_raw  = fig.add_subplot(gs[row, col_])
    if raw_col not in df.columns: continue

    # Normal sample
    norm_samp = df[df['ATTACK_ID']==0].groupby('run_id').head(500)
    for aid in [0] + attack_ids:
        sub = df[df['ATTACK_ID']==aid]
        if len(sub) == 0: continue
        samp = sub.sample(min(500,len(sub)), random_state=42)
        col_data = samp[mean_col].values if mean_col in samp.columns else samp[raw_col].values
        std_data = samp[std_col].values  if std_col  in samp.columns else np.zeros(len(samp))
        x = np.arange(len(col_data))
        lbl  = ATTACK_NAMES[aid]
        col3 = PALETTE[aid]
        ax_raw.plot(x, col_data, lw=1.2, alpha=0.85, color=col3, label=lbl)
        ax_raw.fill_between(x, col_data-std_data, col_data+std_data,
                            alpha=0.12, color=col3)

    ax_raw.set_title(f"{label} — Rolling Mean±Std (w={WINDOW})",
                     fontweight='bold', fontsize=10)
    ax_raw.legend(fontsize=7.5, loc='best')
    ax_raw.set_xlabel("Sample"); ax_raw.grid(alpha=0.3)

plt.suptitle("Temporal Window Statistics: Normal vs Attack Classes",
             fontweight='bold', fontsize=14)
plt.savefig(PLOTS_DIR/"13_temporal_windows.png", dpi=150, bbox_inches='tight')
plt.show()


## 14 · Radar Chart — Model Comparison

In [ ]:
import matplotlib.patheffects as pe

# Compute per-model metrics
metrics_data = {}
all_results = {
    'RF Multi':  (rf_mc,  X_te_sc, y_te),
    'XGB Multi': (xgb_mc, X_te_sc, y_te),
    'MLP Multi': (None,   None,    y_te),
    'RF Binary': (rf_bin, X_te_sc, yb_te),
    'XGB Binary':(xgb_bin,X_te_sc, yb_te),
}
mlp_bin_mdl = SWaTMLP(X_bal.shape[1],[256,128,64],2).to(DEVICE)
try:
    mlp_bin_mdl.load_state_dict(torch.load(MODELS_DIR/"mlp_binary_best.pt", map_location=DEVICE))
    mlp_bin_mdl.eval()
    with torch.no_grad():
        mlp_bin_preds = mlp_bin_mdl(torch.FloatTensor(X_te_sc).to(DEVICE)).argmax(1).cpu().numpy()
except: mlp_bin_preds = np.zeros(len(y_te), dtype=int)

for mname, (mdl, Xeval, ytrue) in all_results.items():
    if mdl is None and 'MLP Multi' in mname:
        ypred = best_preds_mc; ytrue_ = y_te
    elif mname == 'RF Binary':
        ypred = rf_bin.predict(X_te_sc); ytrue_ = yb_te
    elif mname == 'XGB Binary':
        ypred = xgb_bin.predict(X_te_sc); ytrue_ = yb_te
    else:
        ypred = mdl.predict(Xeval); ytrue_ = ytrue

    is_bin = 'Binary' in mname
    metrics_data[mname] = {
        'Accuracy':   accuracy_score(ytrue_, ypred),
        'Macro F1':   f1_score(ytrue_, ypred, average='macro', zero_division=0),
        'Recall':     f1_score(ytrue_, ypred, average='macro', zero_division=0),
        'Precision':  f1_score(ytrue_, ypred, average='weighted', zero_division=0),
    }

categories = list(list(metrics_data.values())[0].keys())
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9,9), subplot_kw=dict(polar=True))
colors_radar = ['#1565C0','#E65100','#2E7D32','#AD1457','#37474F']
for k, (mname, vals) in enumerate(metrics_data.items()):
    vals_list = list(vals.values()) + [list(vals.values())[0]]
    ax.plot(angles, vals_list, lw=2, color=colors_radar[k], label=mname,
            marker='o', markersize=5)
    ax.fill(angles, vals_list, alpha=0.05, color=colors_radar[k])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
ax.set_title("Model Comparison Radar Chart", fontweight='bold',
             fontsize=14, pad=20)
ax.grid(alpha=0.35)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"14_radar.png", dpi=150, bbox_inches='tight')
plt.show()


## 15 · Save Best Model Bundle

In [ ]:
# ── Identify best multi-class model by F1 ────────────────────────────────────
scores = {
    'XGBoost':       f1_score(y_te, xgb_preds,      average='macro', zero_division=0),
    'Random Forest': f1_score(y_te, rf_preds,        average='macro', zero_division=0),
    'MLP':           f1_score(y_te, best_preds_mc,   average='macro', zero_division=0),
}
best_name = max(scores, key=scores.get)
best_model = {'XGBoost': xgb_mc, 'Random Forest': rf_mc}

print("Model F1 scores:")
for n,v in sorted(scores.items(), key=lambda x:-x[1]):
    star = " <-- BEST" if n==best_name else ""
    print(f"  {n}: {v:.4f}{star}")

# ── Save best model with all metadata ────────────────────────────────────────
bundle = {
    'model':        best_model.get(best_name, xgb_mc),
    'scaler':       scaler,
    'feature_cols': FEATURE_COLS,
    'attack_names': ATTACK_NAMES,
    'id_remap':     ID_REMAP,
    'model_name':   best_name,
    'macro_f1':     scores[best_name],
    'num_classes':  NUM_CLASSES,
    'temporal_windows': WINDOW_SIZES,
    'temporal_sensors': TEMPORAL_SENSORS,
    'dead_cols':    DEAD_COLS,
    'training_info': {
        'dataset':          DATA_PATH,
        'total_rows':       int(len(df)),
        'smote_targets':    {2:4000, 3:4000, 7:4000},
        'leakage_window_s': 4.0,
        'test_size':        0.20,
        'random_state':     42,
    }
}
joblib.dump(bundle, MODELS_DIR / "best_model_bundle.joblib", compress=3)
sz = (MODELS_DIR/"best_model_bundle.joblib").stat().st_size/1024/1024
print(f"\nSaved best_model_bundle.joblib  ({sz:.2f} MB)")
print("Bundle contains: model + scaler + feature_cols + all metadata")

# ── Save individual model copies with timestamps ──────────────────────────────
import datetime
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
joblib.dump(xgb_mc,  MODELS_DIR / f"xgb_multiclass_{ts}.joblib", compress=3)
joblib.dump(rf_mc,   MODELS_DIR / f"rf_multiclass_{ts}.joblib",  compress=3)
joblib.dump(xgb_bin, MODELS_DIR / f"xgb_binary_{ts}.joblib",     compress=3)
joblib.dump(rf_bin,  MODELS_DIR / f"rf_binary_{ts}.joblib",      compress=3)
print(f"\nVersioned snapshots saved with timestamp {ts}")

# ── Print all saved plots ─────────────────────────────────────────────────────
print("\n=== All plots saved ===")
for p in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {p.name:<45s}  {p.stat().st_size/1024:.0f} KB")
